# Out-of-distribution generalisation analysis

This file includes the code required to perform an out-of-distribution (OOD) analysis on LF-7T-CycleGAN.

**Important**: This notebook should be run using the conda "mri" environment used for this project.s

### Step 1: Import libraries and Setup

In [1]:
import os
import glob
import shutil

In [ ]:
# These paths will need to be updated to point to the cirrect locations

# Path to OOD dataset
OOD_DATASET_BASE = "C:\\Users\\edwar\\Desktop\\Code\\Dissertation\LF-7T-CycleGAN\\out-of-distribution-analysis\\ood-dataset"

# Paths to trained ensemble models
T1w_ENSEMBLE_DIR = "C:\\Users\\edwar\\Desktop\\Code\\Dissertation\\LF-7T-CycleGAN\\pytorch-cyclegan-and-pix2pix\\checkpoints\\T1w-ensemble"
T2w_ENSEMBLE_DIR = "C:\\Users\\edwar\\Desktop\\Code\\Dissertation\\LF-7T-CycleGAN\\pytorch-cyclegan-and-pix2pix\\checkpoints\\T2w-ensemble"

# Path to save OOD results
PREDICTION_RESULTS_DIR = "C:\\Users\\edwar\\Desktop\\Code\\Dissertation\LF-7T-CycleGAN\\out-of-distribution-analysis\\ood-dataset-results"

# OOD experiment names
T1w_TEST_NAME = "T1w-test"
T2w_TEST_NAME = "T2w-test"

# Maximum number of samples to use from the OOD dataset
MAX_DATASET_SIZE = 50

# Individual paths to OOD datasets within the base directory
OOD_DATASET_T1w_DIR = os.path.join(OOD_DATASET_BASE, "T1w/axial")
OOD_DATASET_T2w_DIR = os.path.join(OOD_DATASET_BASE, "T2w/axial")

### Step 2: Generate OOD dataset using the MRI pre-processor

*Prerequisite:* The `testB` and `trainB` dataset folders in the OOD dataset directory will need to be populated for LF-7T-CycleGAN to run. These will not be used for this analysis but are necessary since the model expects test inputs from both domains. For this study, the standard `testB` and `trainB` folders from the standard T1w and T2ws datasets were used as placeholders.

In [ ]:
# Generate T1w OOD dataset using the pre-processor

!python "../pre-process/preprocess.py" --config-path ood-pre-process-config.json --m4raw-only --save-pngs


==============================  T1w  ==============================
M4RawPreProcessor - Processing 183 scans
Processing 146 scans for dataset type: train
Processing scan #1 : 2022062405
Processing scan #2 : 2022092001
Processing scan #3 : 2023053002
Processing scan #4 : 2022070601
Processing scan #5 : 2022090303
Processing scan #6 : 2022091501
Processing scan #7 : 2022091506
Processing scan #8 : 2022062617
Processing scan #9 : 2022091502
Processing scan #10 : 2022091405
Processing scan #11 : 2022091504
Processing scan #12 : 2022090902
Processing scan #13 : 2022101101
Processing scan #14 : 2022092007
Processing scan #15 : 2022062402
Processing scan #16 : 2022070508
Processing scan #17 : 2022090205
Processing scan #18 : 2022091401
Processing scan #19 : 2022090305
Processing scan #20 : 2022062705
Processing scan #21 : 2022061501
Processing scan #22 : 2022083102
Processing scan #23 : 2022090104
Processing scan #24 : 2022061405
Processing scan #25 : 2022090209
Processing scan #26 : 2022061

### Step 3: Generate OOD predictions

Run inference on the OOD dataset for both contrasts.

In [ ]:
# Generate predictions for T1w OOD dataset

!python "./../pytorch-cyclegan-and-pix2pix/test.py" --dataroot {OOD_DATASET_T1w_DIR}  --test_name {T1w_TEST_NAME} --max_dataset_size {MAX_DATASET_SIZE} --results_dir {PREDICTION_RESULTS_DIR} --model cycle_gan --no_dropout --netG mri --input_nc 1 --output_nc 1 --no_flip --dataset_mode mri --checkpoints_dir {T1w_ENSEMBLE_DIR}  --ensemble_models desktop-200-epochs-ensemble-10 desktop-200-epochs-ensemble-2 desktop-200-epochs-ensemble-3 desktop-200-epochs-ensemble-5 desktop-200-epochs-ensemble-6 desktop-200-epochs-ensemble-8 sarah-desktop-200-epochs-ensemble-7 skynet-200-epochs-ensemble-1 skynet-200-epochs-ensemble-4 skynet-200-epochs-ensemble-9

----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: C:\Users\edwar\Desktop\Code\Dissertation\LF-7T-CycleGAN\pytorch-cyclegan-and-pix2pix\checkpoints\T1w-ensemble	[default: ./checkpoints]
                crop_size: 256                           
                 dataroot: C:\Users\edwar\Desktop\Code\Dissertation\LF-7T-CycleGAN\out-of-distribution-analysis\ood-dataset\T1w/axial	[default: None]
             dataset_mode: mri                           	[default: unaligned]
                direction: AtoB                          
          display_winsize: 256                           
          ensemble_models: ['desktop-200-epochs-ensemble-10', 'desktop-200-epochs-ensemble-2', 'desktop-200-epochs-ensemble-3', 'desktop-200-epochs-ensemble-5', 'desktop-200-epochs-ensemble-6', 'desktop-200-epochs-ensemble-8', 'sarah-desktop-200-epochs-ensemble-7', 'skynet-200

In [4]:
# Generate predictions for T2w OOD dataset

!python "./../pytorch-cyclegan-and-pix2pix/test.py" --dataroot {OOD_DATASET_T2w_DIR}  --test_name {T2w_TEST_NAME} --max_dataset_size {MAX_DATASET_SIZE} --results_dir {PREDICTION_RESULTS_DIR} --model cycle_gan --no_dropout --netG mri --input_nc 1 --output_nc 1 --no_flip --dataset_mode mri --checkpoints_dir {T2w_ENSEMBLE_DIR}  --ensemble_models T2w-axial-ensemble-1 T2w-axial-ensemble-10 T2w-axial-ensemble-2 T2w-axial-ensemble-3 T2w-axial-ensemble-4 T2w-axial-ensemble-5 T2w-axial-ensemble-6 T2w-axial-ensemble-7 T2w-axial-ensemble-8 T2w-axial-ensemble-9

^C


### 4. Save the OOD predictions

Extracts the 0.3T to 7T predictions from the results directory and copies them to `T1w_OUPUT_DIR` and `T1w_OUPUT_DIR`.

In [ ]:
# Create a directory for each OOD prediction result for T1w test set alongside the original LF image

T1w_RESULTS_DIR = os.path.join(PREDICTION_RESULTS_DIR, T1w_TEST_NAME, "test_latest", "images")

print(T1w_RESULTS_DIR)

real_A_images = glob.glob(os.path.join(T1w_RESULTS_DIR, "*real_A*"))
fake_B_images = glob.glob(os.path.join(T1w_RESULTS_DIR, "*fake_B*"))

T1w_OUPUT_DIR = os.path.join(PREDICTION_RESULTS_DIR, "T1w-preds")

os.makedirs(T1w_OUPUT_DIR, exist_ok=True)

for fake_a, fake_b in zip(sorted(real_A_images), sorted(fake_B_images)):
    shutil.copy(fake_a, T1w_OUPUT_DIR)
    shutil.copy(fake_b, T1w_OUPUT_DIR)

print(f"Copied {len(real_A_images)} fake 0.3T images and {len(fake_B_images)} 7T predictions")

C:\Users\edwar\Desktop\Code\Dissertation\LF-7T-CycleGAN\out-of-distribution-analysis\ood-dataset-results\T1w-test\test_latest\images
Copied 50 fake 0.3T images and 50 7T predictions


In [ ]:
# Create a directory for each OOD prediction result for T2w test set alongside the original LF image

T2w_RESULTS_DIR = os.path.join(PREDICTION_RESULTS_DIR, T2w_TEST_NAME, "test_latest", "images")

print(T2w_RESULTS_DIR)

real_A_images = glob.glob(os.path.join(T2w_RESULTS_DIR, "*real_A*"))
fake_B_images = glob.glob(os.path.join(T2w_RESULTS_DIR, "*fake_B*"))

T2w_OUPUT_DIR = os.path.join(PREDICTION_RESULTS_DIR, "T2w-preds")

os.makedirs(T2w_OUPUT_DIR, exist_ok=True)

for fake_a, fake_b in zip(sorted(real_A_images), sorted(fake_B_images)):
    shutil.copy(fake_a, T2w_OUPUT_DIR)
    shutil.copy(fake_b, T2w_OUPUT_DIR)

print(f"Copied {len(real_A_images)} fake 0.3T images and {len(fake_B_images)} 7T predictions")

C:\Users\edwar\Desktop\Code\Dissertation\LF-7T-CycleGAN\out-of-distribution-analysis\ood-dataset-results\T2w-test\test_latest\images
Copied 16 fake 0.3T images and 16 7T predictions
